In [1]:
import numpy as np

In [2]:
# A fake datapoint of input and output

# inputs
x1 = 44
x2 = -0.321
x3 = -493
x4 = 432.382

# Output
y1 = 10

In [3]:
def relu(x):
    return max(0, x)

In [4]:
class Neuron:
    """Class Neuron which accepts a set of inputs and weights.
    It performs the dot product of the inputs and weights and has it's own activation function (ReLU in our case)
    Each neuron has it's own activation function and we return the activation function output.
    """
    def __init__(self, inputs=[], act='relu') -> None:
        self.input_sources = inputs
        self.inputs = np.array([n() if callable(n) else n for n in inputs])
        self.weights = np.random.random(len(self.inputs))
        self.bias = np.random.random()
        self.gradients = np.array([])
        self.act = act

    def return_dot(self):
        # Compute the summation of products of inputs and weights whenever it's called as a function
        dot_prod = np.dot(self.inputs, self.weights) + self.bias
        return dot_prod
    
    def __call__(self):
        # dynamically recompute inputs from upstream sources
        self.inputs = np.array([n() if callable(n) else n for n in self.input_sources])
        out = self.return_dot()
        if self.act == 'relu':
            out = relu(out)
        elif self.act == 'linear':
            pass
        
        return out
    
    def set_input(self, new_inputs):
        self.input_sources = new_inputs
    
    def backward(self,upstream):
        n = self.__call__()         # n(x)
        dot = self.return_dot()     # f(x) inside the n(x)

        df_dx = dot / self.inputs

        # g is supposed to store the differential of the activation function

        g = 1

        if self.act == 'relu':
            g = relu(df_dx * dot)

        self.gradients = self.inputs.copy()   # dn/dw = inputs
        # adding a copy to avoid corrupting original inputs
        self.gradients.resize(self.inputs.size + 1)
        self.gradients[-1] =  1      # dn/db = 1

        self.gradients = upstream * g * self.gradients     # now the f'(x) thiing is multiplied
        # self.gradients is [dn/dw0, dn/dw1, .... , dn/db] computed locally

        # another thing to return, the new upstream: loss gradient with respect to neuron output
        new_upstream = upstream * g * self.weights

        return new_upstream
    
    def step(self, lr):
        new_values = self.weights - lr * self.gradients

        self.weights = new_values[:-1]
        self.bias = new_values[-1]

    def zero_grad(self):
        # reset the gradients to zero to avoid gradient accummulation
        k = self.gradients.size

        self.gradients = np.zeros(k)

    
    def __repr__(self):
        # print out weights and inputs
        inputs = f"Inputs: {self.inputs}"
        weights = f"Weights: {self.weights}"
        bias = f"Bias: {self.bias}"

        final = weights + " * " + inputs + " + " + bias

        return final

In [5]:
np.random.seed(0)

# The forward pass
n1 = Neuron([x1])
print(n1)
n2 = Neuron([n1])
print(n2)
n3 = Neuron([n2])
print(n3)

pred = n3()
print(pred)


Weights: [0.5488135] * Inputs: [44] + Bias: 0.7151893663724195
Weights: [0.60276338] * Inputs: [24.86298354] + Bias: 0.5448831829968969
Weights: [0.4236548] * Inputs: [15.53137908] + Bias: 0.6458941130666561
7.225837400780647


In [6]:
# This is our loss function
# This helps in finding out how our model predictions is close to the actual output
def abs_loss(pred, real):
    # taking the absolute difference (not practical in real-world cases) 
    return abs(pred - real)

# calculate the model loss 
loss = abs_loss(pred, y1)
print(loss)

2.774162599219353


In [7]:
upstream = 0
if pred < y1:
    upstream = -1
elif pred > y1:
    upstream = 1
else:
    upstream = 0

In [8]:
upstream

-1

In [9]:
dl_dn3 = n3.backward(upstream)
print(dl_dn3)

[-1.42422459]


In [10]:
dl_dn2 = n2.backward(dl_dn3)
print(dl_dn2)

[-8.32898604]


In [11]:
dl_dn1 = n1.backward(dl_dn2)
print(dl_dn1)

[-64.2200636]


We have loss gradients for each weight at each neuron. Now we update the old weights.

$w_{n+1} = w_n - \eta \frac{\delta L}{\delta w_n}$

where
- $w_{n+1}$ is the new weight
- $w_n$ is the current weight
- $\eta$ is the learning rate
- $\frac{\delta L}{\delta w_n}$ is the loss gradient with respect to weight

In [12]:
# Take an example learning rate
lr = 10

In [13]:
n1.step(lr)
n2.step(lr)
n3.step(lr)

In [14]:
# putting it all in a loop

# for one single datapoint
x1 = 2
y1 = 4

In [15]:
# forward pass
np.random.seed(0)

# The forward pass
n1 = Neuron([x1])
n2 = Neuron([n1])
n3 = Neuron([n2],act='linear')


def abs_loss(pred, real):
    # taking the absolute difference (not practical in real-world cases) 
    # also return the upstream for first grad calculation
    upstream = 0
    if pred < y1:
        upstream = -1
    elif pred > y1:
        upstream = 1
    else:
        upstream = 0

    return abs(pred - real), upstream

def mean_square_loss(pred, real):
    # returning the mean square error 
    upstream = 2 * (pred - real)
    out = (pred - real) ** 2
    return out, upstream


In [16]:
lr = 0.00001

In [17]:
n = 1000     # number of iterations
for i in range(n):
    pred = n3()
    # calculate the model loss 
    loss, upstream = mean_square_loss(pred, y1)
    # print(loss)

    n1.zero_grad()
    n2.zero_grad()
    n3.zero_grad()

    # perform backpropagation
    dl_dn3 = n3.backward(upstream)
    dl_dn2 = n2.backward(dl_dn3)
    dl_dn1 = n1.backward(dl_dn2)

    # update the model weights
    n1.step(lr)
    n2.step(lr)
    n3.step(lr)
print(loss)

5.995540122525667


In [18]:
n3()

np.float64(1.5520051436168014)

In [19]:
# now creating a dataset
x_train = [i for i in range(1000)]
y_train = [i*i for i in range(1000)]

In [29]:
# define some variables
epochs = 1000
learning_rate = 0.05

In [21]:
n1 = Neuron([x1])
n2 = Neuron([n1])
n3 = Neuron([n2],act='linear')

In [31]:
for epoch in range(epochs):
    for x, y in zip(x_train, y_train):
        n1.set_input([x])
        pred = n3()
        loss, upstream = mean_square_loss(pred, y)
        
        n1.zero_grad()
        n2.zero_grad()
        n3.zero_grad()
        
        dl_dn3 = n3.backward(upstream)
        dl_dn2 = n2.backward(dl_dn3)
        dl_dn1 = n1.backward(dl_dn2)
        
        n1.step(learning_rate)
        n2.step(learning_rate)
        n3.step(learning_rate)
    
print(f"epoch {epoch}, loss: {loss}")
# 692131709555.4279
# 2.4966650020339008e+17

/var/folders/kz/yjf344rd3g33bkkrbyh2vp2m0000gn/T/ipykernel_83384/714927815.py:37: RuntimeWarning: divide by zero encountered in divide
  df_dx = dot / self.inputs


epoch 999, loss: 823444243999.083
